# Lab 1A: Introduction to DuckDB

## 🎯 Learning Objectives

- Understand what DuckDB is and its core characteristics
- Learn why DuckDB matters in modern data analytics
- Identify when to use DuckDB vs. other databases
- Understand DuckDB's place in the data processing ecosystem
- Explore DuckDB use cases and real-world applications
- Understand the data processing flow with DuckDB

## 📋 Prerequisites

- Basic understanding of SQL concepts
- Python 3.8+ installed
- DuckDB Python package installed
- Completed Lab 0: Sample Database Setup

## 🎓 Conceptual Background

### What is DuckDB?

DuckDB is an in-memory analytical database system designed for fast query execution on structured data.

In [ ]:
import duckdb
print(f"DuckDB version: {duckdb.__version__}")

## 🚀 Step 1: Explore DuckDB Characteristics

Let's explore DuckDB's core characteristics and configuration options.

In [ ]:
# Create an in-memory database
con = duckdb.connect(':memory:')

# Check database configuration
config = con.execute("SELECT * FROM duckdb_settings() WHERE name LIKE '%memory%'").fetchall()
print("Memory-related settings:")
for setting in config[:10]:  # Show first 10 settings
    print(f"  {setting[0]}: {setting[1]}")

## 🚀 Step 2: Compare DuckDB Performance

Compare DuckDB with other approaches for simple analytics.

In [ ]:
import pandas as pd
import time

# Create sample data
con.execute("""
    CREATE TABLE test_data AS 
    SELECT * FROM range(1000000) 
    CROSS JOIN (SELECT random() as value FROM range(10))
""")

# DuckDB query
start = time.time()
duckdb_result = con.execute("SELECT AVG(value) FROM test_data").fetchone()
duckdb_time = time.time() - start

# Convert to pandas and query
start = time.time()
df = con.execute("SELECT * FROM test_data").df()
pandas_result = df['value'].mean()
pandas_time = time.time() - start

print(f"DuckDB time: {duckdb_time:.4f}s, Result: {duckdb_result[0]}")
print(f"Pandas time: {pandas_time:.4f}s, Result: {pandas_result}")

## 🚀 Step 3: Practice Data Format Handling

Explore different data format capabilities.

In [ ]:
# Create sample data in different formats
con.execute("""
    CREATE TABLE sample_data AS
    SELECT 
        id,
        'Product ' || id as name,
        random() * 100 as price,
        (random() * 1000)::int as quantity
    FROM range(100)
""")

# Export to different formats
con.execute("COPY sample_data TO 'data/sample.csv' (FORMAT CSV, HEADER)")
con.execute("COPY sample_data TO 'data/sample.parquet' (FORMAT PARQUET)")
con.execute("COPY sample_data TO 'data/sample.json' (FORMAT JSON)")

# Read from different formats
csv_data = con.execute("SELECT COUNT(*) FROM 'data/sample.csv'").fetchone()
parquet_data = con.execute("SELECT COUNT(*) FROM 'data/sample.parquet'").fetchone()
json_data = con.execute("SELECT COUNT(*) FROM 'data/sample.json'").fetchone()

print(f"CSV records: {csv_data[0]}")
print(f"Parquet records: {parquet_data[0]}")
print(f"JSON records: {json_data[0]}")

## 🚀 Step 4: Explore DuckDB's SQL Extensions

Practice DuckDB-specific SQL features.

In [ ]:
# Create sample data for SQL extensions
con.execute("""
    CREATE TABLE sales AS
    SELECT 
        (random() * 100)::int as product_id,
        (random() * 50 + 1)::int as quantity,
        random() * 100 as price,
        ['pending', 'processing', 'shipped', 'delivered'][((random() * 4)::int)] as status,
        '2023-01-01'::date + (random() * 365)::int as order_date
    FROM range(1000)
""")

# GROUP BY ALL (DuckDB extension)
result = con.execute("""
    SELECT product_id, AVG(price), SUM(quantity)
    FROM sales
    GROUP BY ALL
""").fetchall()
print("GROUP BY ALL results:", result[:5])

# Sampling
sample = con.execute("""
    SELECT * FROM sales
    USING SAMPLE 10%
""").fetchdf()
print(f"\nSampled {len(sample)} rows from 1000")

## 💻 Exercise 1: DuckDB Use Case Analysis

Identify appropriate use cases for DuckDB by analyzing different scenarios.

In [ ]:
# For each scenario, determine if DuckDB is a good fit and why

scenarios = [
    "Real-time inventory management for e-commerce",
    "Local data analysis for a data science project",
    "Multi-user business intelligence dashboard",
    "ETL pipeline for data warehouse",
    "Embedded analytics in a desktop application"
]

for scenario in scenarios:
    # Your analysis here
    print(f"Scenario: {scenario}")
    print("DuckDB fit: [Good/Marginal/Poor]")
    print("Reason: [Your reasoning]")
    print()

## 💻 Exercise 2: Data Format Comparison

Compare different data formats for performance and file size.

In [ ]:
import os

# Test different formats
formats = ['csv', 'parquet', 'json']
results = []

for format in formats:
    # Export
    start = time.time()
    con.execute(f"COPY test_data TO 'data/test.{format}' (FORMAT {format.upper()})")
    export_time = time.time() - start
    
    # File size
    file_size = os.path.getsize(f'data/test.{format}') / 1024  # KB
    
    # Query performance
    start = time.time()
    con.execute(f"SELECT COUNT(*), AVG(value) FROM 'data/test.{format}'").fetchone()
    query_time = time.time() - start
    
    results.append({
        'format': format,
        'export_time': export_time,
        'file_size': file_size,
        'query_time': query_time
    })

# Display results
for result in results:
    print(f"{result['format'].upper()}:")
    print(f"  Export: {result['export_time']:.4f}s")
    print(f"  Size: {result['file_size']:.2f}KB")
    print(f"  Query: {result['query_time']:.4f}s")
    print()

## ✅ Verification

Verify your understanding with this comprehensive check.

In [ ]:
print("=== DuckDB Knowledge Verification ===")

# Test 1: Data format handling
con.execute("CREATE TABLE test AS SELECT * FROM range(100)")
con.execute("COPY test TO 'data/verify.csv' (FORMAT CSV, HEADER)")
csv_count = con.execute("SELECT COUNT(*) FROM 'data/verify.csv'").fetchone()[0]
print(f"✓ Data format handling: {csv_count} == 100")

# Test 2: SQL extensions
con.execute("CREATE TABLE sales AS SELECT * FROM range(1000)")
group_result = con.execute("SELECT COUNT(*) FROM sales GROUP BY ALL").fetchall()
print(f"✓ GROUP BY ALL: {len(group_result)} groups")

# Test 3: Performance
start = time.time()
con.execute("SELECT AVG(column0) FROM sales").fetchone()
query_time = time.time() - start
print(f"✓ Performance: Query completed in {query_time:.4f}s")

print("=== All Verification Tests Passed ===")

## 📚 Next Steps

After completing this lab:

1. **Proceed to Lab 2**: Getting Started with DuckDB
2. **Read DuckDB documentation**: https://duckdb.org/docs/
3. **Practice more**: Explore different data formats and SQL extensions